## Visual Artwork Discovery Through Unsupervised Machine Learning

***Project objective:***

The objective of this project is to build an art discovery tool by using a recomendation system to suggest similar artwroks to the ones chosen by the user.

The recomendation system will use features from a public API as well as fetaures engineered directly from the images and then use similarity matrix to give the recomendations. The final interface will be done in Streamlit.
Apart from building the discovery tool, I aim to exmaine how certain features (semantic vs visual) used in the model affect recomendation. 

***Data:***

The data is collected from The Metropolitan Museum of Art Collection API:
https://metmuseum.github.io/

The API provides metadata such as: title, artist, object date, department, medium, classification, culture, tags, and public image URLs.

***This notebook is the initial ETL pipeline and has the following structure:***

Chapter I

- Importing CSV
- Filtering for only 2D artworks
- Creating the initial data frame

Chapter II

- Initial exploratory data analysis
- Checking missing values, available metadata, image availability, artists, dates, mediums, and object types
- Creating a smaple of the too big dataset

Chapter III

- connecting to the API
- Creating two datastes: app_df and ml_df
- Saving the raw and processed metadata files

Chapter IV

- Downloading artwork images from the image URLs

Chapter V

- Feature engineerinf for ML on the ml_df 

Chapter VI

- Recomedation system

In [1]:
import requests
import pandas as pd
import numpy as np
import time
from pathlib import Path

from PIL import Image
from skimage.color import rgb2hsv, rgb2gray
from skimage.feature import canny
from skimage.measure import shannon_entropy

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from collections import Counter

from sklearn.metrics.pairwise import cosine_similarity


In [2]:
RAW_DATA_DIR = Path("/Users/zuzakutowska/Desktop/art_recomender/data/raw")
PROCESSED_DATA_DIR = Path("/Users/zuzakutowska/Desktop/art_recomender/data/processed")
IMAGE_DIR = Path("/Users/zuzakutowska/Desktop/art_recomender/data/images")

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

### Chapter I:
Objective: 
Calling the raw CSV for 2d 

When calling the API I enountered a few challenges. The API would only let me filter by object ID and deprtments. But for my project from all the objects in the database I need only paintings (the best scenarion would be to filter by object type) from as many department as possible (maintaining a big variety of paintings).
Calling each departmetn one by one was too time consuming and returned a lot of failed calles. Thats why I decided to go with a dual approach. Use the API for image URL and the met collection CSV they published on key: object ID.
(the csv has all the fetaures i neede from the API excluding the image url)

This is an ETL pipeline, i willl first filred down the met_df as much as i can and transform it, and only then call the API

In [3]:
met_csv_url = "https://media.githubusercontent.com/media/metmuseum/openaccess/master/MetObjects.csv"

met_df = pd.read_csv(met_csv_url, low_memory=False)

print("Dataset shape:", met_df.shape)

with pd.option_context('display.max_columns', None):
    display(met_df.head())

Dataset shape: (484956, 54)


,Object Number,Is Highlight,Is Timeline Work,Is Public Domain,Object ID,Gallery Number,Department,AccessionYear,Object Name,Title,Culture,Period,Dynasty,Reign,Portfolio,Constituent ID,Artist Role,Artist Prefix,Artist Display Name,Artist Display Bio,Artist Suffix,Artist Alpha Sort,Artist Nationality,Artist Begin Date,Artist End Date,Artist Gender,Artist ULAN URL,Artist Wikidata URL,Object Date,Object Begin Date,Object End Date,Medium,Dimensions,Credit Line,Geography Type,City,State,County,Country,Region,Subregion,Locale,Locus,Excavation,River,Classification,Rights and Reproduction,Link Resource,Object Wikidata URL,Metadata Date,Repository,Tags,Tags AAT URL,Tags Wikidata URL
0,1979.486.1,False,False,False,1,NaN,The American Wing,1979,Coin,One-dollar Liberty Head Coin,NaN,NaN,NaN,NaN,NaN,16429,Maker,,James Barton Longacre,"American, Delaware County, Pennsylvania 1794–1...",,"Longacre, James Barton",American,1794,1869,NaN,http://vocab.getty.edu/page/ulan/500011409,https://www.wikidata.org/wiki/Q3806459,1853,1853,1853,Gold,Dimensions unavailable,"Gift of Heinz L. Stoppelmann, 1979",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search/1,NaN,NaN,"Metropolitan Museum of Art, New York, NY",NaN,NaN,NaN
1,1980.264.5,False,False,False,2,NaN,The American Wing,1980,Coin,Ten-dollar Liberty Head Coin,NaN,NaN,NaN,NaN,NaN,107,Maker,,Christian Gobrecht,1785–1844,,"Gobrecht, Christian",American,1785,1844,NaN,http://vocab.getty.edu/page/ulan/500077295,https://www.wikidata.org/wiki/Q5109648,1901,1901,1901,Gold,Dimensions unavailable,"Gift of Heinz L. Stoppelmann, 1980",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search/2,NaN,NaN,"Metropolitan Museum of Art, New York, NY",NaN,NaN,NaN
2,67.265.9,False,False,False,3,NaN,The American Wing,1967,Coin,Two-and-a-Half Dollar Coin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1909–27,1909,1927,Gold,Diam. 11/16 in. (1.7 cm),"Gift of C. Ruxton Love Jr., 1967",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search/3,NaN,NaN,"Metropolitan Museum of Art, New York, NY",NaN,NaN,NaN
3,67.265.10,False,False,False,4,NaN,The American Wing,1967,Coin,Two-and-a-Half Dollar Coin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1909–27,1909,1927,Gold,Diam. 11/16 in. (1.7 cm),"Gift of C. Ruxton Love Jr., 1967",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search/4,NaN,NaN,"Metropolitan Museum of Art, New York, NY",NaN,NaN,NaN
4,67.265.11,False,False,False,5,NaN,The American Wing,1967,Coin,Two-and-a-Half Dollar Coin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1909–27,1909,1927,Gold,Diam. 11/16 in. (1.7 cm),"Gift of C. Ruxton Love Jr., 1967",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search/5,NaN,NaN,"Metropolitan Museum of Art, New York, NY",NaN,NaN,NaN


In [4]:
met_df.to_csv(
    RAW_DATA_DIR / "raw_met_data.csv",
    index=False
)

In [5]:
met_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 484956 entries, 0 to 484955
Data columns (total 54 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Object Number            484956 non-null  object 
 1   Is Highlight             484956 non-null  bool   
 2   Is Timeline Work         484956 non-null  bool   
 3   Is Public Domain         484956 non-null  bool   
 4   Object ID                484956 non-null  int64  
 5   Gallery Number           49541 non-null   object 
 6   Department               484956 non-null  object 
 7   AccessionYear            481094 non-null  object 
 8   Object Name              482690 non-null  object 
 9   Title                    456153 non-null  object 
 10  Culture                  208190 non-null  object 
 11  Period                   91143 non-null   object 
 12  Dynasty                  23201 non-null   object 
 13  Reign                    11236 non-null   object 
 14  Port

first step is to filter only 2D artowrks.

In [6]:
unique_object_names = met_df["Object Name"].value_counts()
display(unique_object_names)

Object Name
Print                                      102986
Photograph                                  29451
Drawing                                     26018
Book                                        13397
Kylix fragment                               8926
                                            ...  
Oinochoe in the form of a head of a man         1
Diskos                                          1
Neck-amphora, Tyrrhenian, fragmentary           1
Relief fragments, 12                            1
etching                                         1
Name: count, Length: 28631, dtype: int64

if i take just "Painting" from object type and Is Public domain i got 3k results so i need to dig deeper and find other artwroks marked as a different object name.

I always keep in mind that i need the biggest varietty for many cultures and periods for my model.

I will look for:
Painting
Painting, miniature
Watercolor
Drawing
Hanging scroll
Handscroll
Album leaf
Folio from an illustrated manuscript
Bark painting

In [ ]:
keywords = [
    "painting",
    "watercolor",
    "drawing",
    "scroll",
    "album leaf",
    "folio",
    "manuscript"
]

pattern = "|".join(keywords)

relevant_object_names = (
    met_df["Object Name"]
    .dropna()
    .loc[met_df["Object Name"].dropna().str.lower().str.contains(pattern, regex=True)]
    .value_counts()
)

display(relevant_object_names.head(100))

Object Name
Drawing                                              26018
Painting                                              6014
Hanging scroll                                        1618
Drawing Ornament & Architecture                       1516
Painting, miniature                                    948
Drawing ; Ornament and Architecture                    917
Watercolor                                             764
Drawing ; Ornament and architecture                    724
Folio from an illustrated manuscript                   537
Drawings, Ornament & Architecture                      511
Folio                                                  483
Album leaf                                             377
Bark painting                                          338
Handscroll                                             337
Portfolio                                              311
Drawing Ornament and Architecture                      288
Folding fan mounted as an album leaf        

In [8]:
selected_object_names = [
    "Painting",
    "Watercolor",
    "Drawing",
    "Drawings",
    "Hanging scroll",
    "Hanging scrolls",
    "Handscroll",
    "Handscrolls",
    "Album leaf",
    "Illustrated album leaf",
    "Illustrated  album leaf",
    "Folio from an illustrated manuscript",
    "Bark painting",
    "Wall painting"
]

In [9]:
met_df = met_df[
    (met_df["Object Name"].isin(selected_object_names)) &
    (met_df["Is Public Domain"] == True)
].copy()

also noticed that some of the artowrks appeared to be 2D but in "medium" had materilas that did not match with 2D, im cleaning those

In [10]:
exclude_3d_medium_keywords = [
    "ivory", "porcelain", "ceramic", "terracotta", "marble",
    "alabaster", "glass", "enamel", "bronze", "copper",
    "silver", "brass", "steel", "leather", "mother-of-pearl",
    "shell", "bone", "horn", "jewelry", "snuffbox"
]

met_df["medium_clean"] = (
    met_df["Medium"]
    .fillna("")
    .str.lower()
    .str.strip()
)

two_d_artworks_df = met_df[
    ~met_df["medium_clean"].str.contains(
        "|".join(exclude_3d_medium_keywords),
        regex=True,
        na=False
    )
].copy()

print("Before filtering:", met_df.shape)
print("After filtering:", two_d_artworks_df.shape)

Before filtering: (20729, 55)
After filtering: (20358, 55)


In [11]:
with pd.option_context('display.max_columns', None):
    display(two_d_artworks_df.head(30))

,Object Number,Is Highlight,Is Timeline Work,Is Public Domain,Object ID,Gallery Number,Department,AccessionYear,Object Name,Title,Culture,Period,Dynasty,Reign,Portfolio,Constituent ID,Artist Role,Artist Prefix,Artist Display Name,Artist Display Bio,Artist Suffix,Artist Alpha Sort,Artist Nationality,Artist Begin Date,Artist End Date,Artist Gender,Artist ULAN URL,Artist Wikidata URL,Object Date,Object Begin Date,Object End Date,Medium,Dimensions,Credit Line,Geography Type,City,State,County,Country,Region,Subregion,Locale,Locus,Excavation,River,Classification,Rights and Reproduction,Link Resource,Object Wikidata URL,Metadata Date,Repository,Tags,Tags AAT URL,Tags Wikidata URL,medium_clean
413,83.2.235,False,False,True,472,NaN,The American Wing,1883,Painting,Portrait of Benjamin Franklin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776–1883,1776,1883,Panel,Diam. 3 1/8 in. (7.9 cm),"Gift of William H. Huntington, 1883",Made in,NaN,NaN,NaN,France,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search...,https://www.wikidata.org/wiki/Q79024193,NaN,"Metropolitan Museum of Art, New York, NY",Benjamin Franklin|Men|Portraits,http://vocab.getty.edu/page/ulan/500331804|htt...,https://www.wikidata.org/wiki/Q34969|https://w...,panel
415,83.2.239,False,False,True,474,NaN,The American Wing,1883,Painting,Plaque Portrait of Benjamin Franklin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776–1883,1776,1883,Oil on cardboard,Diam. 5 1/8 in. (13 cm),"Gift of William H. Huntington, 1883",Made in,NaN,NaN,NaN,France,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search...,https://www.wikidata.org/wiki/Q79023962,NaN,"Metropolitan Museum of Art, New York, NY",Benjamin Franklin|Men|Portraits,http://vocab.getty.edu/page/ulan/500331804|htt...,https://www.wikidata.org/wiki/Q34969|https://w...,oil on cardboard
423,83.2.234,False,False,True,482,NaN,The American Wing,1883,Painting,Portrait of Benjamin Franklin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776–1883,1776,1883,Oil on wood,5 5/8 x 3 1/2 in. (14.3 x 8.9 cm),"Gift of William H. Huntington, 1883",Made in,NaN,NaN,NaN,France,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search...,https://www.wikidata.org/wiki/Q79023693,NaN,"Metropolitan Museum of Art, New York, NY",Benjamin Franklin|Men|Portraits,http://vocab.getty.edu/page/ulan/500331804|htt...,https://www.wikidata.org/wiki/Q34969|https://w...,oil on wood
424,83.2.236,False,False,True,483,NaN,The American Wing,1883,Painting,Portrait of Benjamin Franklin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776–1883,1776,1883,Panel,5 3/8 x 4 in. (13.7 x 10.2 cm),"Gift of William H. Huntington, 1883",Made in,NaN,NaN,NaN,France,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search...,https://www.wikidata.org/wiki/Q79023574,NaN,"Metropolitan Museum of Art, New York, NY",Benjamin Franklin|Men|Portraits,http://vocab.getty.edu/page/ulan/500331804|htt...,https://www.wikidata.org/wiki/Q34969|https://w...,panel
425,83.2.233,False,False,True,484,NaN,The American Wing,1883,Painting,Medallion of Benjamin Franklin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776–1883,1776,1883,Oil on wood,5 3/4 x 4 1/4 in. (14.6 x 10.8 cm),"Gift of William H. Huntington, 1883",Made in,NaN,NaN,NaN,France,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.metmuseum.org/art/collection/search...,https://www.wikidata.org/wiki/Q79023461,NaN,"Metropolitan Museum of Art, New York, NY",Benjamin Franklin|Men|Profiles,http://vocab.getty.edu/page/ulan/500331804|htt...,https://www.wikidata.org/wiki/Q34969|https://w...,oil on wood
426,83.2.222,False,False,True,485,NaN,The American Wing,1883,Painting,Portrait of Benjamin Franklin,NaN,NaN,NaN,NaN,NaN,120,Artist,Possibly after,H. Sanford,1783–1822,,"Sanford, H.",,1783,1822,NaN,NaN,NaN,1776–1883,1776,1883,Panel,5 3/8 x 4 1/4 in. (13.

In [12]:
two_d_artworks_df.to_csv(
    RAW_DATA_DIR / "two_d_artworks.csv",
    index=False
)

## Chapter II: 
Preparing two clean datsets (for app and ml) and connecting to the API

In [13]:
columns_to_keep = [
    "Object ID",
    "Department",
    "Title",
    "Object Name",
    "Culture",
    "Period",
    "Dynasty",
    "Reign",
    "Artist Display Name",
    "Artist Nationality",
    "Artist Begin Date",
    "Artist End Date",
    "Artist Gender",
    "Object Begin Date",
    "Object End Date",
    "Medium",
    "Country",
    "Link Resource",
    "Is Public Domain",
    "Tags",
    "medium_clean"
]

two_d_artworks_df = two_d_artworks_df[columns_to_keep].copy()

print("Metadata shape:", two_d_artworks_df.shape)

with pd.option_context("display.max_columns", None):
    display(two_d_artworks_df.head())

Metadata shape: (20358, 21)


,Object ID,Department,Title,Object Name,Culture,Period,Dynasty,Reign,Artist Display Name,Artist Nationality,Artist Begin Date,Artist End Date,Artist Gender,Object Begin Date,Object End Date,Medium,Country,Link Resource,Is Public Domain,Tags,medium_clean
413,472,The American Wing,Portrait of Benjamin Franklin,Painting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776,1883,Panel,France,http://www.metmuseum.org/art/collection/search...,True,Benjamin Franklin|Men|Portraits,panel
415,474,The American Wing,Plaque Portrait of Benjamin Franklin,Painting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776,1883,Oil on cardboard,France,http://www.metmuseum.org/art/collection/search...,True,Benjamin Franklin|Men|Portraits,oil on cardboard
423,482,The American Wing,Portrait of Benjamin Franklin,Painting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776,1883,Oil on wood,France,http://www.metmuseum.org/art/collection/search...,True,Benjamin Franklin|Men|Portraits,oil on wood
424,483,The American Wing,Portrait of Benjamin Franklin,Painting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776,1883,Panel,France,http://www.metmuseum.org/art/collection/search...,True,Benjamin Franklin|Men|Portraits,panel
425,484,The American Wing,Medallion of Benjamin Franklin,Painting,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1776,1883,Oil on wood,France,http://www.metmuseum.org/art/collection/search...,True,Benjamin Franklin|Men|Profiles,oil on wood


In [14]:
two_d_artworks_df = two_d_artworks_df.rename(columns={
    "Object ID": "object_id",
    "Department": "department",
    "Title": "title",
    "Object Name": "object_name",
    "Culture": "culture",
    "Period": "period",
    "Dynasty": "dynasty",
    "Reign": "reign",
    "Artist Display Name": "artist_display_name",
    "Artist Nationality": "artist_nationality",
    "Artist Begin Date": "artist_begin_date",
    "Artist End Date": "artist_end_date",
    "Artist Gender": "artist_gender",
    "Object Begin Date": "object_begin_date",
    "Object End Date": "object_end_date",
    "Medium": "medium",
    "Country": "country",
    "Link Resource": "link_resource",
    "Is Public Domain": "is_public_domain",
    "Tags": "tags"
})

missing values

In [15]:
missing_summary = pd.DataFrame({
    "missing_count": two_d_artworks_df.isna().sum(),
    "missing_percent": two_d_artworks_df.isna().mean() * 100,
    "unique_values": two_d_artworks_df.nunique()
}).sort_values("missing_percent", ascending=False)

missing_summary

,missing_count,missing_percent,unique_values
dynasty,20358,100.000000,0
reign,20358,100.000000,0
country,19644,96.492779,42
period,18466,90.706356,114
artist_gender,18122,89.016603,12
culture,15473,76.004519,166
title,1011,4.966107,16628
tags,859,4.219471,8220
artist_begin_date,849,4.170351,1537
artist_display_name,849,4.170351,4990


There is not much we can do about the missing values.
I will drop 2 features that have way too many missing values and the non relevant "is_public_domain" and clean the "artist_gender" feature since in the met database description, there was infomration that if the value is nan, the artist gender is female

In [16]:
columns_to_drop = [
    "dynasty",
    "reign",
    "is_public_domain"
]

two_d_artworks_df = two_d_artworks_df.drop(columns=columns_to_drop)

In [17]:
def clean_artist_gender(value):
    value = str(value).strip()
    
    if "Female" in value:
        return "Female"
    else:
        return "Male"

two_d_artworks_df["artist_gender"] = two_d_artworks_df["artist_gender"].apply(clean_artist_gender)

I noticed that "culture" and "country" both have a lot of missing values and they store information related to the origin country, I need to examine these and look for some kind of overlap.

In [18]:
display(two_d_artworks_df["country"].value_counts(dropna=False))
display(two_d_artworks_df["culture"].value_counts(dropna=False))

country
NaN                                       19644
India                                       277
Iran                                        138
United States                               135
present-day Afghanistan                      26
France                                       21
Turkey                                       10
present-day Pakistan                         10
Spain                                        10
Germany                                       7
present-day Uzbekistan                        7
England                                       7
probably Syria or Iraq                        6
Iran or Iraq                                  6
Iraq                                          5
The Netherlands                               4
Egypt                                         4
Italy                                         4
Mexico                                        4
Italy or the Netherlands                      3
Ireland                         

culture
NaN                                             15473
American                                         2394
Japan                                            1117
China                                             732
Southern and Northern Cheyenne                    105
French, Versailles                                 73
Korea                                              45
Italian                                            21
India, Madurai, Tamil Nadu                         17
Spanish                                            14
Roman                                              13
China (Xinjiang Uyghur Autonomous Region)          11
Netherlandish                                      10
Tibet                                              10
Dutch                                               9
India (Rajasthan)                                   9
Thailand                                            8
French                                              8
British             

my solution to this would be to create dictionaries for country and culture to map the current values to more simple, general regions of origin. Hopefully we will have some overlap and could inferr some of the countries from culture.

In [19]:
country_replacements = {
    "Indiaal": "India",
    "The Netherlands": "Netherlands",
    "Present-day Uzbekistan": "Uzbekistan",
    "present-day Uzbekistan": "Uzbekistan",
    "present-day Afghanistan": "Afghanistan",
    "modern-day Afghanistan": "Afghanistan",
    "present-day Pakistan": "Pakistan",
    "United States|United States": "United States"
}

culture_to_country = {
    "american": "United States",
    "southern and northern cheyenne": "United States",
    "southern arapaho": "United States",
    "arapaho": "United States",
    "japan": "Japan",
    "china": "China",
    "chinese": "China",
    "korea": "Korea",
    "french": "France",
    "italian": "Italy",
    "spanish": "Spain",
    "german": "Germany",
    "dutch": "Netherlands",
    "netherlandish": "Netherlands",
    "flemish": "Belgium",
    "british": "United Kingdom",
    "austrian": "Austria",
    "india": "India",
    "indian": "India",
    "peru": "Peru",
    "peruvian": "Peru",
    "mexican": "Mexico",
    "russian": "Russia",
    "thailand": "Thailand",
    "thai": "Thailand",
    "nepal": "Nepal",
    "tibet": "Tibet",
    "ecuador": "Ecuador",
    "sri lanka": "Sri Lanka",
    "qajar": "Iran",
    "sasanian": "Iran"
}

def infer_country_from_culture(culture):
    if pd.isna(culture):
        return np.nan

    culture = str(culture).lower().strip()

    for keyword, country in culture_to_country.items():
        if keyword in culture:
            return country

    return np.nan

two_d_artworks_df["country_clean"] = (
    two_d_artworks_df["country"]
    .replace(country_replacements)
    .str.strip()
)

two_d_artworks_df["country_from_culture"] = (
    two_d_artworks_df["culture"]
    .apply(infer_country_from_culture)
)

two_d_artworks_df["origin_country"] = (
    two_d_artworks_df["country_clean"]
    .fillna(two_d_artworks_df["country_from_culture"])
)

print("Original country missing:", two_d_artworks_df["country"].isna().sum())
print("Origin country missing:", two_d_artworks_df["origin_country"].isna().sum())
print(
    "Countries inferred from culture:",
    (
        two_d_artworks_df["country"].isna()
        & two_d_artworks_df["origin_country"].notna()
    ).sum()
)

display(
    two_d_artworks_df[
        ["object_id", "country", "culture", "origin_country"]
    ].head(20)
)

Original country missing: 19644
Origin country missing: 14976
Countries inferred from culture: 4668


,object_id,country,culture,origin_country
413,472,France,NaN,France
415,474,France,NaN,France
423,482,France,NaN,France
424,483,France,NaN,France
425,484,France,NaN,France
426,485,France,NaN,France
427,486,China,NaN,China
441,500,Japan,NaN,Japan
875,955,NaN,American,United States
1529,1668,NaN,"American, Shaker",United States


analysing data types

In [20]:
two_d_artworks_df.dtypes

object_id                int64
department              object
title                   object
object_name             object
culture                 object
period                  object
artist_display_name     object
artist_nationality      object
artist_begin_date       object
artist_end_date         object
artist_gender           object
object_begin_date        int64
object_end_date          int64
medium                  object
country                 object
link_resource           object
tags                    object
medium_clean            object
country_clean           object
country_from_culture    object
origin_country          object
dtype: object

artist_begin_date and artist_end_date should be int

In [21]:
date_columns = [
    "artist_begin_date",
    "artist_end_date"
]

for column in date_columns:
    two_d_artworks_df[column] = pd.to_numeric(
        two_d_artworks_df[column],
        errors="coerce"
    ).astype("Int64")

outliers 

the only values i could fix here and look for outliers are the dates
im looking for end dates that are bigger that begin dates also for dates bigger than 2026

*** artist dates include "|" in some entires where there was more that one artist working (for now i will ignore it since this information is used only for display in the app - it will not be included in the set for recomendation system)

In [22]:
invalid_object_dates = two_d_artworks_df[
    (two_d_artworks_df["object_begin_date"] > two_d_artworks_df["object_end_date"]) |
    (two_d_artworks_df["object_begin_date"] == 0) |
    (two_d_artworks_df["object_end_date"] == 0)
]

print("Invalid or zero object dates:", invalid_object_dates.shape)

display(invalid_object_dates[["object_id", "object_begin_date", "object_end_date"]].head())

Invalid or zero object dates: (6, 21)


,object_id,object_begin_date,object_end_date
17143,20891,0,0
62859,72639,0,0
202925,334024,1814,0
247446,379940,0,1860
249903,382521,1785,1780


Since its just 6 rows i will remove the rows with both dates as 0s and adjust the other ones

In [23]:
two_d_artworks_df = two_d_artworks_df[
    ~(
        (two_d_artworks_df["object_begin_date"] == 0) &
        (two_d_artworks_df["object_end_date"] == 0)
    )
].copy()

two_d_artworks_df.loc[
    two_d_artworks_df["object_begin_date"] == 0,
    "object_begin_date"
] = two_d_artworks_df["object_end_date"]

two_d_artworks_df.loc[
    two_d_artworks_df["object_end_date"] == 0,
    "object_end_date"
] = two_d_artworks_df["object_begin_date"]

two_d_artworks_df = two_d_artworks_df[
    two_d_artworks_df["object_begin_date"]
    <= two_d_artworks_df["object_end_date"]
].copy()

The dataset is prepared for next steps. 
But the size of the datset is too big (20729). WE need to sample it smartly. Ideally i want to have paintings from as many periods and regions as possible, but also i would want to look at the missing values 
So my streagy would be to create a new "competion score" for each row and then add a century feature so we can sample the same amount of artwors from each century but also consider the department and i want to first pick the painting with the higest completion

(especially importnat is considering the department, previosly i did sampling just looking at the completion score and century and i got 50% of asian art)

In [ ]:
important_metadata_cols = [
    "title",
    "artist_display_name",
    "artist_nationality",
    "artist_gender",
    "origin_country",
    "object_name",
    "medium",
    "tags",
    "object_begin_date",
    "object_end_date"
]

two_d_artworks_df["metadata_completeness_score"] = (
    two_d_artworks_df[important_metadata_cols]
    .replace("Unknown", pd.NA)
    .notna()
    .sum(axis=1)
)

two_d_artworks_df["metadata_completeness_score"].describe()

count    20355.000000
mean         9.083075
std          0.530421
min          5.000000
25%          9.000000
50%          9.000000
75%          9.000000
max         10.000000
Name: metadata_completeness_score, dtype: float64

In [25]:
two_d_artworks_df["object_mid_year"] = (
    two_d_artworks_df["object_begin_date"] + two_d_artworks_df["object_end_date"]
) / 2

def assign_century(year):
    if pd.isna(year):
        return "Unknown"
    elif year < 0:
        century = int(abs(year) // 100 + 1)
        return f"{century}th century BCE"
    elif year == 0:
        return "1st century"
    else:
        century = int((year - 1) // 100 + 1)
        return f"{century}th century"
    
two_d_artworks_df["century"] = two_d_artworks_df["object_mid_year"].apply(assign_century)
two_d_artworks_df["century"].value_counts().sort_index()

century
10th century          5
11th century         25
12th century         49
13th century         75
14th century        263
15th century        503
16th century       2160
17th century       3183
18th century       4179
19th century       9488
1th century           4
1th century BCE      12
20th century        379
24th century          1
4th century           1
4th century BCE       1
5th century           2
6th century          13
7th century           4
8th century           4
9th century           4
Name: count, dtype: int64

In [26]:
target_size = 2000

sampling_df = two_d_artworks_df.copy()

sampling_df = sampling_df.sort_values(
    by=[
        "century",
        "department",
        "metadata_completeness_score"
    ],
    ascending=[True, True, False]
)

sampling_df["sampling_rank"] = (
    sampling_df
    .groupby(["century", "department"])
    .cumcount()
)

sampling_df = sampling_df.sort_values(
    by=["sampling_rank", "metadata_completeness_score"],
    ascending=[True, False]
)

artworks_sample = (
    sampling_df
    .head(target_size)
    .drop(columns="sampling_rank")
    .reset_index(drop=True)
)

print("Final sample shape:", artworks_sample.shape)

Final sample shape: (2000, 24)


In [27]:
with pd.option_context("display.max_columns", None):
    display(artworks_sample.head(10))

,object_id,department,title,object_name,culture,period,artist_display_name,artist_nationality,artist_begin_date,artist_end_date,artist_gender,object_begin_date,object_end_date,medium,country,link_resource,tags,medium_clean,country_clean,country_from_culture,origin_country,metadata_completeness_score,object_mid_year,century
0,39542,Asian Art,五代 傳董元 溪岸圖 軸|Riverbank,Hanging scroll,China,Southern Tang dynasty (937–76),Dong Yuan,Chinese,930,960,Male,937,976,Hanging scroll; ink and color on silk,NaN,http://www.metmuseum.org/art/collection/search...,Landscapes|Rivers|Mountains,hanging scroll; ink and color on silk,NaN,China,China,10,956.5,10th century
1,39668,Asian Art,"北宋 郭熙 樹色平遠圖 卷 |Old Trees, Level Dist...",Handscroll,China,Northern Song dynasty (960–1127),Guo Xi,Chinese,1000,1090,Male,1070,1090,Handscroll; ink and color on silk,NaN,http://www.metmuseum.org/art/collection/search...,Trees|Landscapes,handscroll; ink and color on silk,NaN,China,China,10,1080.0,11th century
2,451193,Islamic Art,"""Hare"", Folio from the Mantiq al-wahsh (Speech...",Folio from an illustrated manuscript,NaN,NaN,Ka'b al-Ahbar,,<NA>,652,Male,1000,1199,Opaque watercolor on paper,Egypt,http://www.metmuseum.org/art/collection/search...,Rabbits,opaque watercolor on paper,Egypt,NaN,Egypt,10,1099.5,11th century
3,39936,Asian Art,北宋 徽宗 竹禽圖 卷|Finches and bamboo,Handscroll,China,Northern Song dynasty (960–1127),Emperor Huizong,Chinese,1082,1135,Male,1100,1127,Handscroll; ink and color on silk,NaN,http://www.metmuseum.org/art/collection/search...,Birds|Bamboo,handscroll; ink and color on silk,NaN,China,China,10,1113.5,12th century
4,36005,Asian Art,"南宋 夏珪 山市晴嵐圖 冊頁|Mountain Market, Clearing Mist",Album leaf,China,Southern Song dynasty (1127–1279),Xia Gui,Chinese,1195,1230,Male,1200,1230,Album leaf; ink on silk \r\n,NaN,http://www.metmuseum.org/art/collection/search...,Landscapes|Mountains,album leaf; ink on silk,NaN,China,China,10,1215.0,13th century
5,446288,Islamic Art,"""Physician Preparing an Elixir"", Folio from a ...",Folio from an illustrated manuscript,NaN,NaN,'Abdullah ibn al-Fadl,,<NA>,<NA>,Male,1199,1249,"Ink, opaque watercolor, and gold on paper",Iraq or Northern Jazira,http://www.metmuseum.org/art/collection/search...,Men|Arabic|Medicine,"ink, opaque watercolor, and gold on paper",Iraq or Northern Jazira,NaN,Iraq or Northern Jazira,10,1224.0,13th century
6,459046,Robert Lehman Collection,Saints Bartholomew and Simon,Painting,"Italian, Umbria",NaN,Master of Saint Francis,Italian,1250,1275,Male,1250,1275,Tempera and gold on wood,NaN,http://www.metmuseum.org/art/collection/search...,Men|Saints,tempera and gold on wood,NaN,Italy,Italy,10,1262.5,13th century
7,36029,Asian Art,수월관음도 고려|水月觀音圖 高麗|Water-moon Avalokiteshvara,Hanging scroll,Korea,Goryeo dynasty (918–1392),Unidentified artist,,<NA>,<NA>,Male,1300,1349,Hanging scroll; ink and color on silk,NaN,http://www.metmuseum.org/art/collection/search...,Bodhisattvas|Buddhism|Avalokiteshvara,hanging scroll; ink and color on silk,NaN,Korea,Korea,10,1324.5,14th century
8,447071,Islamic Art,Folio from a Mu'nis al-ahrar fi daqa'iq al-ash...,Folio from an illustrated manuscript,NaN,NaN,Muhammad ibn Badr al-Din Jajarmi,Iranian,1315,1365,Male,1315,1365,"Ink, opaque watercolor, and gold on paper",Iran,http://www.metmuseum.org/art/collection/search...,Poetry|Bulls|Crabs|Rams|Zodiac|Lions,"ink, opaque watercolor, and gold on paper",Iran,NaN,Iran,10,1340.0,14th century
9,458955,Robert Lehman Collection,Saint Peter,Painting,"Italian, Siena",NaN,Follower of Barna da Siena|Follower of Lippo M...,Sienese|Italian,<NA>,<NA>,Male,1336,1370,"Tempera on wood, gold ground",NaN,http://www.metmuseum.org/art/collection/search...,Saint Peter,"tempera on wood, gold ground",NaN,Italy,Italy,10,1353.0,14th century


In [28]:
artworks_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   object_id                    2000 non-null   int64  
 1   department                   2000 non-null   object 
 2   title                        1950 non-null   object 
 3   object_name                  2000 non-null   object 
 4   culture                      883 non-null    object 
 5   period                       456 non-null    object 
 6   artist_display_name          1801 non-null   object 
 7   artist_nationality           1801 non-null   object 
 8   artist_begin_date            1516 non-null   Int64  
 9   artist_end_date              1519 non-null   Int64  
 10  artist_gender                2000 non-null   object 
 11  object_begin_date            2000 non-null   int64  
 12  object_end_date              2000 non-null   int64  
 13  medium            

## Chapter III: Connecting to the api and creating the app datset and the ML datset

In [29]:
image_rows = []
failed_api_requests = []

for i, row in artworks_sample.iterrows():
    object_id = row["object_id"]
    
    api_url = f"https://collectionapi.metmuseum.org/public/collection/v1/objects/{object_id}"
    
    try:
        response = requests.get(api_url, timeout=20)
        
        if response.status_code == 200:
            item = response.json()
            
            image_rows.append({
                "object_id": object_id,
                "primary_image": item.get("primaryImage"),
                "primary_image_small": item.get("primaryImageSmall")
            })
        else:
            failed_api_requests.append({
                "object_id": object_id,
                "status_code": response.status_code
            })
    
    except Exception as e:
        failed_api_requests.append({
            "object_id": object_id,
            "status_code": None,
            "error": str(e)
        })
    
    if i % 50 == 0:
        print(f"Checked {i} objects")
    
    time.sleep(0.5)

image_urls_df = pd.DataFrame(image_rows)
failed_api_df = pd.DataFrame(failed_api_requests)

print("Image URLs collected:", image_urls_df.shape)
print("Failed API requests:", failed_api_df.shape)

Checked 0 objects
Checked 50 objects
Checked 100 objects
Checked 150 objects
Checked 200 objects
Checked 250 objects
Checked 300 objects
Checked 350 objects
Checked 400 objects
Checked 450 objects
Checked 500 objects
Checked 550 objects
Checked 600 objects
Checked 650 objects
Checked 700 objects
Checked 750 objects
Checked 800 objects
Checked 850 objects
Checked 900 objects
Checked 950 objects
Checked 1000 objects
Checked 1050 objects
Checked 1100 objects
Checked 1150 objects
Checked 1200 objects
Checked 1250 objects
Checked 1300 objects
Checked 1350 objects
Checked 1400 objects
Checked 1450 objects
Checked 1500 objects
Checked 1550 objects
Checked 1600 objects
Checked 1650 objects
Checked 1700 objects
Checked 1750 objects
Checked 1800 objects
Checked 1850 objects
Checked 1900 objects
Checked 1950 objects
Image URLs collected: (1991, 3)
Failed API requests: (9, 3)


In [30]:
image_urls_df.to_csv(
    PROCESSED_DATA_DIR / "image_urls.csv",
    index=False
)

In [31]:
image_urls_df.head()

,object_id,primary_image,primary_image_small
0,39542,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...
1,39668,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...
2,451193,https://images.metmuseum.org/CRDImages/is/orig...,https://images.metmuseum.org/CRDImages/is/web-...
3,39936,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...
4,36005,https://images.metmuseum.org/CRDImages/as/orig...,https://images.metmuseum.org/CRDImages/as/web-...


Now we can merge the URLs with the sample dataset (using inner join)

In [32]:
artworks_with_images = artworks_sample.merge(
    image_urls_df,
    on="object_id",
    how="inner",
    validate="one_to_one"
)

print("Merged dataset shape:", artworks_with_images.shape)

Merged dataset shape: (1991, 26)


now we will create two separate datsets:
1. for the app
2. for recomendation system

App dataset

In [ ]:
artworks_with_images = pd.read_csv()

In [ ]:
app_df = artworks_with_images[["object_id", "department", "title", "object_name", 
                              "origin_country", "period", "artist_display_name", 
                              "artist_begin_date", "artist_end_date", "medium",
                              "link_resource", "tags", "century", "primary_image", "artist_gender"]].copy()

app_df.head()

,object_id,department,title,object_name,origin_country,period,artist_display_name,artist_begin_date,artist_end_date,medium,link_resource,tags,century,primary_image
0,39542,Asian Art,五代 傳董元 溪岸圖 軸|Riverbank,Hanging scroll,China,Southern Tang dynasty (937–76),Dong Yuan,930,960,Hanging scroll; ink and color on silk,http://www.metmuseum.org/art/collection/search...,Landscapes|Rivers|Mountains,10th century,https://images.metmuseum.org/CRDImages/as/orig...
1,39668,Asian Art,"北宋 郭熙 樹色平遠圖 卷 |Old Trees, Level Dist...",Handscroll,China,Northern Song dynasty (960–1127),Guo Xi,1000,1090,Handscroll; ink and color on silk,http://www.metmuseum.org/art/collection/search...,Trees|Landscapes,11th century,https://images.metmuseum.org/CRDImages/as/orig...
2,451193,Islamic Art,"""Hare"", Folio from the Mantiq al-wahsh (Speech...",Folio from an illustrated manuscript,Egypt,NaN,Ka'b al-Ahbar,<NA>,652,Opaque watercolor on paper,http://www.metmuseum.org/art/collection/search...,Rabbits,11th century,https://images.metmuseum.org/CRDImages/is/orig...
3,39936,Asian Art,北宋 徽宗 竹禽圖 卷|Finches and bamboo,Handscroll,China,Northern Song dynasty (960–1127),Emperor Huizong,1082,1135,Handscroll; ink and color on silk,http://www.metmuseum.org/art/collection/search...,Birds|Bamboo,12th century,https://images.metmuseum.org/CRDImages/as/orig...
4,36005,Asian Art,"南宋 夏珪 山市晴嵐圖 冊頁|Mountain Market, Clearing Mist",Album leaf,China,Southern Song dynasty (1127–1279),Xia Gui,1195,1230,Album leaf; ink on silk \r\n,http://www.metmuseum.org/art/collection/search...,Landscapes|Mountains,13th century,https://images.metmuseum.org/CRDImages/as/orig...


For displaying in the app i am filling in all missing text values with "Unknown"

In [ ]:
text_columns = ["department", "title", "object_name", "origin_country", "period", "artist_display_name", "medium", "tags", "century"]

app_df[text_columns] = app_df[text_columns].fillna("Unknown")

The app_df is ready and can be saved

In [35]:
app_df.to_csv(
    PROCESSED_DATA_DIR / "app_artworks_metadata.csv",
    index=False
)

print("Saved app_df to:")
print(PROCESSED_DATA_DIR / "app_artworks_metadata.csv")

Saved app_df to:
/Users/zuzakutowska/Desktop/art_recomender/data/processed/app_artworks_metadata.csv


ML dataset

In [36]:
ml_df = artworks_with_images[["object_id", "department", "object_name", "origin_country",
                              "artist_gender", "tags", "medium_clean", "object_mid_year", "primary_image"]].copy()

ml_df.head()

,object_id,department,object_name,origin_country,artist_gender,tags,medium_clean,object_mid_year,primary_image
0,39542,Asian Art,Hanging scroll,China,Male,Landscapes|Rivers|Mountains,hanging scroll; ink and color on silk,956.5,https://images.metmuseum.org/CRDImages/as/orig...
1,39668,Asian Art,Handscroll,China,Male,Trees|Landscapes,handscroll; ink and color on silk,1080.0,https://images.metmuseum.org/CRDImages/as/orig...
2,451193,Islamic Art,Folio from an illustrated manuscript,Egypt,Male,Rabbits,opaque watercolor on paper,1099.5,https://images.metmuseum.org/CRDImages/is/orig...
3,39936,Asian Art,Handscroll,China,Male,Birds|Bamboo,handscroll; ink and color on silk,1113.5,https://images.metmuseum.org/CRDImages/as/orig...
4,36005,Asian Art,Album leaf,China,Male,Landscapes|Mountains,album leaf; ink on silk,1215.0,https://images.metmuseum.org/CRDImages/as/orig...


The Ml datset needs feature engineering, this will be done after we download images. since we willl also be extracting features from the images and adding them to the dataset.
Thats why im not saving this df yet.

just final quick check if the datasets have the same amount of rows, and we wont have any lost artwoks.

In [37]:
print(app_df.shape)
print(ml_df.shape)

merged_try = app_df.merge(
    ml_df,
    on="object_id",
    how="inner",
    validate="one_to_one"
)

print(merged_try.shape)

(1991, 14)
(1991, 9)
(1991, 22)


## Chapter IV: Downloading images

In [38]:
download_results = []

for _, row in ml_df.iterrows():
    object_id = row["object_id"]
    image_url = row["primary_image"]
    image_path = IMAGE_DIR / f"{object_id}.jpg"

    try:
        if image_path.exists():
            status = "already_exists"
        else:
            response = requests.get(image_url, timeout=30)
            response.raise_for_status()

            with open(image_path, "wb") as file:
                file.write(response.content)

            status = "downloaded"

    except Exception as error:
        status = f"failed: {error}"

    download_results.append({
        "object_id": object_id,
        "local_image_path": str(image_path),
        "download_status": status
    })

    time.sleep(0.1)

download_log = pd.DataFrame(download_results)

print(download_log["download_status"].value_counts())
print("Images saved in:", IMAGE_DIR)

download_status
downloaded                                                                                                          1979
failed: Invalid URL '': No scheme supplied. Perhaps you meant https://?                                                3
failed: HTTPSConnectionPool(host='images.metmuseum.org', port=443): Read timed out.                                    1
failed: 404 Client Error: Not Found for url: https://images.metmuseum.org/CRDImages/ep/original/DP375525.jpg           1
failed: 404 Client Error: Not Found for url: https://images.metmuseum.org/CRDImages/as/original/DP153535.jpg           1
failed: 404 Client Error: Not Found for url: https://images.metmuseum.org/CRDImages/ep/original/DP-12431-001.jpg       1
failed: 404 Client Error: Not Found for url: https://images.metmuseum.org/CRDImages/ma/original/sf1984.548.1.JPG       1
failed: 404 Client Error: Not Found for url: https://images.metmuseum.org/CRDImages/ma/original/sf1984.548.2.JPG       1
failed: 404 Clie

After the image dowanload, I need to make sure that both dfs still have the same artworks and all had a succesful image download.

In [39]:
successful_downloads = download_log[
    download_log["download_status"].isin(["downloaded", "already_exists"])
][["object_id", "local_image_path"]]

ml_df = ml_df.merge(
    successful_downloads,
    on="object_id",
    how="inner"
)

app_df = app_df[
    app_df["object_id"].isin(successful_downloads["object_id"])
].copy()


app_df.to_csv(
    PROCESSED_DATA_DIR / "app_artworks_metadata.csv",
    index=False
)

print("Final ml_df shape:", ml_df.shape)
print("Final app_df shape:", app_df.shape)
print("Updated app_df saved.")

Final ml_df shape: (1979, 10)
Final app_df shape: (1979, 14)
Updated app_df saved.


## Chapter V: Feature Engineering

Extracting new features from images, encoding and sclaing ml_df 

Images in Pandas are a 2D array
The first added feature will examin the overall colour balance of the artpiece.
For element (pixel) of the 2D array we will calculte the everage colour intensity.

Extracted Visual Features

* Mean RGB values – average red, green, and blue intensity across all pixels.
* RGB standard deviation – variation in each colour channel.
* Mean saturation – how vivid or muted the artwork is.
* Mean brightness – overall lightness of the image.
* Grayscale standard deviation – simple measure of tonal contrast.
* Dark pixel ratio – proportion of very dark pixels.
* Light pixel ratio - proportion of very bright pixels.
* Edge density – proportion of pixels identified as edges, representing line and detail intensity.
* Entropy – level of pixel variation and visual complexity.
* Sharpness – amount of fine detail and intensity change between neighbouring pixels.
* Aspect ratio – relationship between image width and height, indicating portrait, landscape, or scroll-like format.

In [ ]:
def extract_image_features(image_path):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    aspect_ratio = width / height

    image_array = np.array(image, dtype=np.float32) / 255.0
    hsv_image = rgb2hsv(image_array)
    grayscale_image = rgb2gray(image_array)
    edges = canny(grayscale_image)

    hue = hsv_image[:, :, 0]
    saturation = hsv_image[:, :, 1]
    brightness = hsv_image[:, :, 2]

    hue_histogram, _ = np.histogram(hue, bins=12, range=(0, 1))
    hue_histogram = hue_histogram / hue_histogram.sum()

    saturation_histogram, _ = np.histogram(saturation,bins=5,range=(0, 1))
    saturation_histogram = (saturation_histogram / saturation_histogram.sum())
    
    brightness_histogram, _ = np.histogram(brightness, bins=5, range=(0, 1))
    brightness_histogram = (brightness_histogram / brightness_histogram.sum())
    
    grayscale_uint8 = (grayscale_image * 255).astype(np.uint8)

    features = {
        "mean_red": image_array[:, :, 0].mean(),
        "mean_green": image_array[:, :, 1].mean(),
        "mean_blue": image_array[:, :, 2].mean(),

        "std_red": image_array[:, :, 0].std(),
        "std_green": image_array[:, :, 1].std(),
        "std_blue": image_array[:, :, 2].std(),

        "mean_saturation": saturation.mean(),
        "mean_brightness": brightness.mean(),

        "grayscale_std": grayscale_image.std(),
        "dark_pixel_ratio": (grayscale_image < 0.20).mean(),
        "light_pixel_ratio": (grayscale_image > 0.80).mean(),

        "edge_density": edges.mean(),
        "entropy": shannon_entropy(grayscale_uint8),

        "aspect_ratio": aspect_ratio
    }

    for index, value in enumerate(hue_histogram):
        features[f"hue_bin_{index}"] = value

    for index, value in enumerate(saturation_histogram):
        features[f"saturation_bin_{index}"] = value

    for index, value in enumerate(brightness_histogram):
        features[f"brightness_bin_{index}"] = value

    return pd.Series(features)

In [41]:
image_features = ml_df["local_image_path"].apply(
    extract_image_features
)

ml_df = pd.concat(
    [
        ml_df.reset_index(drop=True),
        image_features.reset_index(drop=True)
    ],
    axis=1
)

/Users/zuzakutowska/myenv/lib/python3.13/site-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (100815600 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


In [42]:
ml_df.to_csv(
    PROCESSED_DATA_DIR / "ml_artworks_metadata.csv",
    index=False
)

The ml_df is ready.

I habe decided to divide the art features into teo groups:
- sematic 
- visual

I was to be able to control the weight of both of these categories in my recomendation system. 
The user can udjust if they want artworks similar visually or form the same period, culture etc.

In [43]:
semantic_df = ml_df[["object_id", "department", "object_name", "origin_country", "artist_gender", "tags", "medium_clean", "object_mid_year"]].copy()

visual_df = ml_df[["object_id", "mean_red", "mean_green", "mean_blue", "std_red", "std_green", "std_blue",
                   "mean_saturation", "mean_brightness", "grayscale_std", "dark_pixel_ratio", "light_pixel_ratio", "edge_density", "entropy", "aspect_ratio",
                   "hue_bin_0", "hue_bin_1", "hue_bin_2", "hue_bin_3", "hue_bin_4", "hue_bin_5", "hue_bin_6", "hue_bin_7", "hue_bin_8", "hue_bin_9", "hue_bin_10",
                   "hue_bin_11", "saturation_bin_0", "saturation_bin_1", "saturation_bin_2", "saturation_bin_3", "saturation_bin_4", "brightness_bin_0", "brightness_bin_1",
                   "brightness_bin_2", "brightness_bin_3", "brightness_bin_4"]].copy()


Now I will appropriately scale and encode features inthese dfs in order to prepare them for ML.

Starting with visual_df, here features are mostly numerical. I will scale them accordingly.

In [44]:
visual_feature_columns = visual_df.columns.drop("object_id")

visual_scaler = MinMaxScaler()

visual_df[visual_feature_columns] = visual_scaler.fit_transform(
    visual_df[visual_feature_columns]
)

In [45]:
visual_df[visual_feature_columns].describe().round(2)

,mean_red,mean_green,mean_blue,std_red,std_green,std_blue,mean_saturation,mean_brightness,grayscale_std,dark_pixel_ratio,...,saturation_bin_0,saturation_bin_1,saturation_bin_2,saturation_bin_3,saturation_bin_4,brightness_bin_0,brightness_bin_1,brightness_bin_2,brightness_bin_3,brightness_bin_4
count,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00,...,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00,1979.00
mean,0.58,0.52,0.45,0.37,0.36,0.36,0.35,0.58,0.35,0.13,...,0.38,0.36,0.19,0.07,0.03,0.11,0.15,0.20,0.30,0.27
std,0.22,0.22,0.22,0.17,0.16,0.16,0.18,0.22,0.16,0.21,...,0.34,0.28,0.20,0.12,0.09,0.19,0.17,0.18,0.24,0.31
min,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.42,0.35,0.27,0.25,0.24,0.24,0.23,0.42,0.24,0.00,...,0.10,0.12,0.01,0.00,0.00,0.00,0.02,0.07,0.10,0.02
50%,0.61,0.54,0.45,0.37,0.35,0.34,0.35,0.61,0.35,0.01,...,0.25,0.32,0.13,0.01,0.00,0.01,0.08,0.16,0.24,0.13
75%,0.76,0.70,0.61,0.48,0.46,0.45,0.46,0.75,0.45,0.17,...,0.60,0.57,0.31,0.07,0.01,0.13,0.23,0.29,0.47,0.48
max,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


Means are close to 0 and stadard deviation is 1, looks good.

In [46]:
visual_df.to_csv(
    PROCESSED_DATA_DIR / "visual_features.csv",
    index=False
)

sematic_df requires more work.

I will:
- binary encode artist_gender, 
- one-hot encode department and object_name,
- multi-hot encode origin_country (by looking for keywords (countries))
- multi-hot encode tags (by looking for most popular keywords)
- multi-hot encode medium_clean (by looking for most common mediums)
- scale numerical object_mid_year


In [47]:
semantic_df.head()

,object_id,department,object_name,origin_country,artist_gender,tags,medium_clean,object_mid_year
0,39542,Asian Art,Hanging scroll,China,Male,Landscapes|Rivers|Mountains,hanging scroll; ink and color on silk,956.5
1,39668,Asian Art,Handscroll,China,Male,Trees|Landscapes,handscroll; ink and color on silk,1080.0
2,451193,Islamic Art,Folio from an illustrated manuscript,Egypt,Male,Rabbits,opaque watercolor on paper,1099.5
3,39936,Asian Art,Handscroll,China,Male,Birds|Bamboo,handscroll; ink and color on silk,1113.5
4,36005,Asian Art,Album leaf,China,Male,Landscapes|Mountains,album leaf; ink on silk,1215.0


In [ ]:
semantic_df["artist_gender_encoded"] = (
    semantic_df["artist_gender"]
    .map({
        "male": 0,
        "Male": 0,
        "female": 1,
        "Female": 1 # did i lower idk
    })
)

for object_name i noticed that there were some names that resabled the same thing, i will map them 

In [49]:
semantic_df["object_name_clean"] = (
    semantic_df["object_name"]
    .str.lower()
    .str.strip()
    .replace({
        "drawings": "drawing",
        "hanging scrolls": "hanging scroll",
        "handscrolls": "handscroll",
        "illustrated  album leaf": "illustrated album leaf"
    })
)

object_name_encoded = pd.get_dummies(
    semantic_df["object_name_clean"],
    prefix="object_name",
    dtype=int
)

semantic_df = pd.concat(
    [semantic_df, object_name_encoded],
    axis=1
)

In [50]:
department_encoded = pd.get_dummies(
    semantic_df["department"],
    prefix="department",
    dtype=int
)

semantic_df = pd.concat(
    [semantic_df, department_encoded],
    axis=1
)

In [51]:
country_keywords = [ # how annoying was this but idk a way around it without manually listing them?
    "china",
    "united states",
    "japan",
    "india",
    "iran",
    "france",
    "italy",
    "afghanistan",
    "netherlands",
    "spain",
    "germany",
    "korea",
    "belgium",
    "united kingdom",
    "syria",
    "iraq",
    "egypt",
    "pakistan",
    "tibet",
    "peru",
    "tunisia",
    "mexico",
    "thailand",
    "austria",
    "uzbekistan",
    "turkey",
    "nepal"
]

country_encoded = pd.DataFrame(
    index=semantic_df.index
)

for country in country_keywords:
    column_name = "country_" + country.replace(" ", "_")

    country_encoded[column_name] = (
        semantic_df["origin_country"]
        .str.lower()
        .str.contains(country, regex=False, na=False)
        .astype(int)
    )

semantic_df = pd.concat(
    [semantic_df, country_encoded],
    axis=1
)

the "tags" feature is quite messy
i will first create lists with all the mentioned tags and take the 60 most common tags 


(i am a bit worried they might overpower the other features but also the content of the artpiece is very important)

In [52]:
semantic_df["tags_list"] = (
    semantic_df["tags"]
    .fillna("")
    .apply(lambda x: x.lower().split("|"))
)

tag_counts = Counter()

for tags in semantic_df["tags_list"]:
    for tag in tags:
        tag = tag.strip()

        if tag != "":
            tag_counts[tag] += 1
       
     
top_60_tags = [
    tag
    for tag, count in tag_counts.most_common(60)
]

for tag in top_60_tags:
    column_name = "tag_" + tag.replace(" ", "_")

    semantic_df[column_name] = semantic_df["tags_list"].apply(
        lambda tags: 1 if tag in [item.strip() for item in tags] else 0
    )

In [53]:
medium_keywords = [
    "oil",
    "ink",
    "watercolor",
    "tempera",
    "graphite",
    "chalk",
    "charcoal",
    "pencil",
    "crayon",
    "fresco",
    "pigment",
    "gold",
    "canvas",
    "paper",
    "wood",
    "silk",
    "plaster"
]

for medium in medium_keywords:
    column_name = "medium_" + medium

    semantic_df[column_name] = (
        semantic_df["medium_clean"]
        .str.lower()
        .str.contains(medium, regex=False, na=False)
        .astype(int)
    )

In [54]:
year_scaler = StandardScaler()

semantic_df["object_mid_year_scaled"] = year_scaler.fit_transform(
    semantic_df[["object_mid_year"]]
)

final check of the semantic_df and we can drop columns pre encoding

In [55]:
with pd.option_context('display.max_columns', None):
    display(semantic_df.head())

,object_id,department,object_name,origin_country,artist_gender,tags,medium_clean,object_mid_year,artist_gender_encoded,object_name_clean,object_name_album leaf,object_name_drawing,object_name_folio from an illustrated manuscript,object_name_handscroll,object_name_hanging scroll,object_name_illustrated album leaf,object_name_painting,object_name_wall painting,object_name_watercolor,department_Ancient Near Eastern Art,department_Arms and Armor,"department_Arts of Africa, Oceania, and the Americas",department_Asian Art,department_Drawings and Prints,department_European Paintings,department_European Sculpture and Decorative Arts,department_Greek and Roman Art,department_Islamic Art,department_Medieval Art,department_Modern and Contemporary Art,department_Musical Instruments,department_Robert Lehman Collection,department_The American Wing,department_The Cloisters,country_china,country_united_states,country_japan,country_india,country_iran,country_france,country_italy,country_afghanistan,country_netherlands,country_spain,country_germany,country_korea,country_belgium,country_united_kingdom,country_syria,country_iraq,country_egypt,country_pakistan,country_tibet,country_peru,country_tunisia,country_mexico,country_thailand,country_austria,country_uzbekistan,country_turkey,country_nepal,tags_list,tag_men,tag_women,tag_landscapes,tag_portraits,tag_horses,tag_mountains,tag_madonna_and_child,tag_trees,tag_birds,tag_flowers,tag_christ,tag_angels,tag_female_nudes,tag_saints,tag_dogs,tag_virgin_mary,tag_boats,tag_buddhism,tag_ornament,tag_soldiers,tag_animals,tag_bodhisattvas,tag_boys,tag_swords,tag_saint_john_the_baptist,tag_lions,tag_human_figures,tag_kings,tag_musical_instruments,tag_crucifixion,tag_books,tag_buildings,tag_infants,tag_working,tag_girls,tag_rivers,tag_houses,tag_interiors,tag_poetry,tag_cows,tag_heads,tag_calligraphy,tag_male_nudes,tag_saint_john_the_evangelist,tag_firearms,tag_elephants,tag_bamboo,tag_buddha,tag_deer,tag_jesus,tag_hunting,tag_still_life,tag_leaves,tag_putti,tag_armor,tag_bow_and_arrow,tag_musicians,tag_battles,tag_children,tag_profiles,medium_oil,medium_ink,medium_watercolor,medium_tempera,medium_graphite,medium_chalk,medium_charcoal,medium_pencil,medium_crayon,medium_fresco,medium_pigment,medium_gold,medium_canvas,medium_paper,medium_wood,medium_silk,medium_plaster,object_mid_year_scaled
0,39542,Asian Art,Hanging scroll,China,Male,Landscapes|Rivers|Mountains,hanging scroll; ink and color on silk,956.5,0,hanging scroll,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"[landscapes, rivers, mountains]",0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,-2.236373
1,39668,Asian Art,Handscroll,China,Male,Trees|Landscapes,handscroll; ink and color on silk,1080.0,0,handscroll,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"[trees, landscapes]",0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,-1.810848
2,451193,Islamic Art,Folio from an illustrated manuscript,Egypt,Male,Rabbits,opaque watercolor on paper,1099.5,0,folio from an illustrated manuscript,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,[rabbits],0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,-1.743660
3,39936,Asian Art,Handscroll,China,Male,Birds|Bamboo,handscroll; ink and color on silk,1113.5,0,handscroll,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"[birds, bamboo]",0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,-1.695422
4

In [56]:
semantic_df.to_csv(
    PROCESSED_DATA_DIR / "semantic_features_full.csv",
    index=False
)

In [57]:
semantic_df = semantic_df.drop(columns=["department", "object_name", "origin_country", "medium_clean", "tags_list", "artist_gender", "tags", "object_mid_year", "object_name_clean"])

### Chapter I: Recomedation system

We will use two  similarity matrices 

The high idea is that i use the  similarity matrix to get the most similar artworks and then get the ids of these artowrks and use the app_df to get all the info about them for display in the app

First need to make sure we have the same objects for indexes in the two dfs: semantic_df and visual_Df

In [58]:
semantic_df = semantic_df.reset_index(drop=True)
visual_df = visual_df.reset_index(drop=True)

In [59]:
print("semantic_df rows:", len(semantic_df))
print("visual_df rows:", len(visual_df))

semantic_df rows: 1979
visual_df rows: 1979


In [60]:
semantic_df["object_id"].equals(visual_df["object_id"])

True

To move on with the recomendation system i need to separate the "object_id"s from the set for similarity matrix

In [61]:
object_ids = semantic_df["object_id"].copy()

X_visual = visual_df.drop(columns="object_id")

X_semantic = semantic_df.select_dtypes(include="number").drop(
    columns="object_id"
)

In [62]:
visual_similarity = cosine_similarity(X_visual)
semantic_similarity = cosine_similarity(X_semantic)

In [63]:
print(visual_similarity.shape)
print(semantic_similarity.shape)

(1979, 1979)
(1979, 1979)


In [64]:
np.save(
    PROCESSED_DATA_DIR / "visual_similarity.npy",
    visual_similarity.astype(np.float32)
)

np.save(
    PROCESSED_DATA_DIR / "semantic_similarity.npy",
    semantic_similarity.astype(np.float32)
)

np.save(
    PROCESSED_DATA_DIR / "object_ids.npy",
    object_ids.to_numpy()
)

In [ ]:
def get_recommendations(
    selected_object_id,
    semantic_weight,
    app_df,
    object_ids,
    visual_similarity,
    semantic_similarity,
    top_n=10
):
    
    selected_index = np.where(
        object_ids == selected_object_id
    )[0][0]

    visual_weight = 1 - semantic_weight # in the app will get the values form user

    combined_scores = (
        semantic_weight * semantic_similarity[selected_index]
        + visual_weight * visual_similarity[selected_index]
    )

    recommended_indices = combined_scores.argsort()[::-1]

    recommended_indices = recommended_indices[
        recommended_indices != selected_index
    ]

    recommended_indices = recommended_indices[:top_n]

    recommendations = pd.DataFrame({
        "object_id": object_ids[recommended_indices],
        "similarity_score": combined_scores[recommended_indices]
    })

    recommendations = recommendations.merge(
        app_df,
        on="object_id",
        how="left"
    )

    return recommendations

Test the recomendation matrix

In [66]:
selected_id = object_ids[0]

test_recommendations = get_recommendations(
    selected_object_id=selected_id,
    semantic_weight=0,
    app_df=app_df,
    object_ids=object_ids,
    visual_similarity=visual_similarity,
    semantic_similarity=semantic_similarity,
    top_n=10
)

test_recommendations

,object_id,similarity_score,department,title,object_name,origin_country,period,artist_display_name,artist_begin_date,artist_end_date,medium,link_resource,tags,century,primary_image
0,435817,0.947019,European Paintings,The Crucifixion with the Virgin and Saint John,Painting,Unknown,Unknown,Hendrick ter Brugghen,1588,1629,Oil on canvas,http://www.metmuseum.org/art/collection/search...,Skulls|Saint John the Evangelist|Crucifixion|C...,17th century,https://images.metmuseum.org/CRDImages/ep/orig...
1,435573,0.946725,European Paintings,Flora and Zephyr,Painting,Unknown,Unknown,Jacopo Amigoni,1682,1752,Oil on canvas,http://www.metmuseum.org/art/collection/search...,Goddess|Putti|Flowers|Landscapes,18th century,https://images.metmuseum.org/CRDImages/ep/orig...
2,435713,0.946027,European Paintings,The Disillusioned Medea,Painting,Unknown,Unknown,Paulus Bor,1601,1669,Oil on canvas,http://www.metmuseum.org/art/collection/search...,Female Nudes,17th century,https://images.metmuseum.org/CRDImages/ep/orig...
3,459083,0.938808,Robert Lehman Collection,Burgomaster Jan van Duren (1613–1687),Painting,Unknown,Unknown,Gerard ter Borch the Younger,1617,1681,Oil on canvas,http://www.metmuseum.org/art/collection/search...,Men|Portraits,17th century,https://images.metmuseum.org/CRDImages/rl/orig...
4,436119,0.938454,European Paintings,The Rope Dance,Painting,Unknown,Unknown,Léonard Defrance,1735,1805,Oil on wood,http://www.metmuseum.org/art/collection/search...,Men|Women|Spectators|Dancing,18th century,https://images.metmuseum.org/CRDImages/ep/orig...
5,436216,0.934781,European Paintings,Portrait of a Young Woman as a Vestal Virgin,Painting,Unknown,Unknown,François Hubert Drouais,1727,1775,Oil on canvas,http://www.metmuseum.org/art/collection/search...,Portraits|Women|Flowers,18th century,https://images.metmuseum.org/CRDImages/ep/orig...
6,435728,0.933487,European Paintings,The Last Communion of Saint Jerome,Painting,Unknown,Unknown,Botticelli (Alessandro di Mariano Filipepi),1444,1510,Tempera and gold on wood,http://www.metmuseum.org/art/collection/search...,Saint Jerome|Monks|Crucifixion,15th century,https://images.metmuseum.org/CRDImages/ep/orig...
7,435829,0.931261,European Paintings,Echo,Painting,Unknown,Unknown,Alexandre Cabanel,1823,1889,Oil on canvas,http://www.metmuseum.org/art/collection/search...,Nymphs|Female Nudes,19th century,https://images.metmuseum.org/CRDImages/ep/orig...
8,435756,0.930955,European Paintings,A Classical Landscape,Painting,Unknown,Unknown,Sébastien Bourdon,1616,1671,Oil on canvas,http://www.metmuseum.org/art/collection/search...,Horses|Landscapes|Human Figures,17th century,https://images.metmuseum.org/CRDImages/ep/orig...
9,435688,0.929460,European Paintings,Portrait of a Woman,Painting,Unknown,Unknown,Louis Léopold Boilly,1761,1845,Oil on canvas,http://www.metmuseum.org/art/collection/search...,Portraits|Women,19th century,https://images.metmuseum.org/CRDImages/ep/orig...


Final sumary and check

In [67]:
print("App data:", app_df.shape)
print("Visual data:", visual_df.shape)
print("Semantic data:", semantic_df.shape)
print("Visual similarity:", visual_similarity.shape)
print("Semantic similarity:", semantic_similarity.shape)

print(
    "IDs aligned:",
    visual_df["object_id"].equals(semantic_df["object_id"])
)

App data: (1979, 14)
Visual data: (1979, 37)
Semantic data: (1979, 131)
Visual similarity: (1979, 1979)
Semantic similarity: (1979, 1979)
IDs aligned: True
